In [1]:
# Batch inference on multiple images
def batch_inference(image_paths, model, device, batch_size=4, conf_threshold=0.4):
    """
    Perform batch inference on multiple thermal images.
    
    Args:
        image_paths: List of paths to thermal images
        model: Thermal detection model
        device: torch.device
        batch_size: Number of images per batch
        conf_threshold: Confidence threshold for detections
    
    Returns:
        List of detection results for each image
    """
    results = []
    model.eval()
    
    # Process in batches
    for batch_start in range(0, len(image_paths), batch_size):
        batch_end = min(batch_start + batch_size, len(image_paths))
        batch_paths = image_paths[batch_start:batch_end]
        batch_tensors = []
        
        # Load and preprocess images
        for img_path in batch_paths:
            pil_img = Image.open(img_path).convert('L')
            preprocessed, _, _ = preprocess_thermal_image(pil_img, target_size=512)
            batch_tensors.append(preprocessed)
        
        # Stack batch
        batch_input = torch.stack(batch_tensors).to(device)
        
        # Forward pass
        with torch.no_grad():
            batch_predictions = model(batch_input)
        
        # Parse each image's predictions
        for i, pred in enumerate(batch_predictions if isinstance(batch_predictions, list) else [batch_predictions]):
            detections = parse_predictions(pred, conf_threshold=conf_threshold)
            results.append({
                'path': batch_paths[i],
                'detections': detections
            })
    
    return results

# Run batch inference on first 5 images
print("Running batch inference on multiple thermal images...")
num_images = min(5, len(available_images))
batch_results = batch_inference(available_images[:num_images], student_model, device, batch_size=2)

print(f"\n=== Batch Inference Results ({num_images} images) ===")
for i, result in enumerate(batch_results, 1):
    img_name = Path(result['path']).name
    num_dets = len(result['detections']['scores'])
    print(f"{i}. {img_name:60s} → {num_dets} detections")
    for score, cls_id in zip(result['detections']['scores'], result['detections']['classes']):
        cls_name = result['detections']['class_names'][cls_id]
        print(f"   - {cls_name:8s}: {score:.4f}")

Running batch inference on multiple thermal images...


NameError: name 'available_images' is not defined

## Optional: Batch Inference on Multiple Images

Process multiple thermal images in a single batch for faster inference.

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Original thermal image
ax = axes[0]
thermal_np = np.array(thermal_pil)
im1 = ax.imshow(thermal_np, cmap='hot')
ax.set_title('Original Thermal Image', fontsize=14, fontweight='bold')
ax.set_xlabel(f'Width: {thermal_np.shape[1]} px')
ax.set_ylabel(f'Height: {thermal_np.shape[0]} px')
plt.colorbar(im1, ax=ax, label='Pixel Value')

# Plot 2: Preprocessed image with detections overlaid
ax = axes[1]
preprocessed_np = preprocessed_tensor.squeeze(0).cpu().numpy()
im2 = ax.imshow(preprocessed_np, cmap='hot')
ax.set_title(f'Detections (Preprocessed {imgsz}x{imgsz})', fontsize=14, fontweight='bold')
ax.set_xlabel(f'Width: {imgsz} px')
ax.set_ylabel(f'Height: {imgsz} px')

# Draw bounding boxes
colors = ['red', 'lime', 'blue', 'yellow', 'cyan', 'magenta']
if len(detections['scores']) > 0:
    for i, (bbox, score, cls_id) in enumerate(zip(detections['bboxes'], 
                                                     detections['scores'], 
                                                     detections['classes'])):
        x, y, w, h = bbox
        # Convert from center-based (x, y, w, h) to corner-based (left, top)
        left = x - w / 2
        top = y - h / 2
        
        color = colors[i % len(colors)]
        rect = Rectangle((left, top), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Add label
        cls_name = detections['class_names'][cls_id] if cls_id < len(detections['class_names']) else f"class_{cls_id}"
        label = f"{cls_name} {score:.2f}"
        ax.text(left, top - 5, label, color=color, fontsize=10, fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.3'))

plt.colorbar(im2, ax=ax, label='Pixel Value')
plt.tight_layout()
plt.show()

print("\n✓ Visualization completed")

## Section 7: Visualize Detection Results

Display the thermal image with overlaid detection results including bounding boxes and confidence scores.

In [ ]:
# Post-process predictions
def parse_predictions(predictions, conf_threshold=0.5):
    """
    Parse model predictions into bounding boxes and scores.
    
    Handles different prediction formats from YOLOv8Thermal model.
    """
    detections = {
        'bboxes': [],      # [x, y, w, h] format
        'scores': [],      # Confidence scores
        'classes': [],     # Class IDs
        'class_names': ['person', 'car', 'bicycle']
    }
    
    # Extract predictions based on format
    if isinstance(predictions, dict):
        if 'predictions' in predictions:
            preds_tensor = predictions['predictions']
        else:
            # Try to find prediction tensor
            for v in predictions.values():
                if isinstance(v, torch.Tensor) and len(v.shape) == 3:
                    preds_tensor = v
                    break
    elif isinstance(predictions, (list, tuple)):
        for p in predictions:
            if isinstance(p, torch.Tensor) and len(p.shape) == 3:
                preds_tensor = p
                break
    else:
        preds_tensor = predictions
    
    # Move to CPU and convert to numpy
    if isinstance(preds_tensor, torch.Tensor):
        preds_tensor = preds_tensor.cpu().numpy()
    
    print(f"Predictions tensor shape: {preds_tensor.shape}")
    
    # For YOLOv8 format: (batch, num_detections, 6) where 6 = [x, y, w, h, conf, class]
    # or (batch, num_anchors, num_classes + 5)
    if len(preds_tensor.shape) == 3:
        batch_preds = preds_tensor[0]  # Take first batch
        
        # Handle detection format [x, y, w, h, conf, class]
        if batch_preds.shape[1] == 6:
            for detection in batch_preds:
                x, y, w, h, conf, cls_id = detection
                if conf > conf_threshold:
                    detections['bboxes'].append([x, y, w, h])
                    detections['scores'].append(conf)
                    detections['classes'].append(int(cls_id))
        
        # Handle format with per-class confidence (common in YOLOv8)
        elif batch_preds.shape[1] > 5:
            for detection in batch_preds:
                x, y, w, h = detection[:4]
                conf = detection[4]
                cls_scores = detection[5:]
                
                if conf > conf_threshold:
                    cls_id = np.argmax(cls_scores)
                    cls_conf = cls_scores[cls_id]
                    detections['bboxes'].append([x, y, w, h])
                    detections['scores'].append(max(conf, cls_conf))
                    detections['classes'].append(int(cls_id))
    
    # Convert to numpy arrays
    if detections['bboxes']:
        detections['bboxes'] = np.array(detections['bboxes'])
        detections['scores'] = np.array(detections['scores'])
        detections['classes'] = np.array(detections['classes'])
    
    return detections

# Parse predictions
conf_threshold = 0.4
detections = parse_predictions(predictions, conf_threshold=conf_threshold)

print(f"\n=== Detection Results ===")
print(f"Detections found: {len(detections['scores'])}")
print(f"Confidence threshold: {conf_threshold}")

if len(detections['scores']) > 0:
    print("\nDetailed Results:")
    for i, (bbox, score, cls_id) in enumerate(zip(detections['bboxes'], 
                                                     detections['scores'], 
                                                     detections['classes'])):
        cls_name = detections['class_names'][cls_id] if cls_id < len(detections['class_names']) else f"class_{cls_id}"
        x, y, w, h = bbox
        print(f"  [{i+1}] {cls_name:8s} | Score: {score:.4f} | BBox: ({x:.1f}, {y:.1f}, {w:.1f}, {h:.1f})")

## Section 6: Post-process Detection Output

Extract and post-process model outputs including bounding boxes, confidence scores, and class predictions.

In [ ]:
# Set model to evaluation mode
student_model.eval()

# Forward pass without gradient computation
with torch.no_grad():
    print("Running forward pass...")
    predictions = student_model(input_batch)
    print("✓ Forward pass completed")

# Handle predictions output
if isinstance(predictions, dict):
    print("Model output type: Dictionary")
    print(f"Keys: {predictions.keys()}")
    for key, value in predictions.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {type(value)}")
elif isinstance(predictions, (list, tuple)):
    print(f"Model output type: {type(predictions).__name__} with {len(predictions)} elements")
    for i, pred in enumerate(predictions):
        if isinstance(pred, torch.Tensor):
            print(f"  Element {i}: Tensor {pred.shape}")
else:
    print(f"Model output type: {type(predictions)}")
    if isinstance(predictions, torch.Tensor):
        print(f"Output shape: {predictions.shape}")

## Section 5: Forward Pass Through Student Model

Set model to evaluation mode and perform forward pass without gradient computation.

In [ ]:
# Preprocessing function for thermal images
def preprocess_thermal_image(pil_image, target_size=512, pad_value=0.0):
    """
    Preprocess thermal image: convert to tensor, normalize, and apply letterbox padding.
    
    Args:
        pil_image: PIL Image (L mode, single channel)
        target_size: Target image size (assumes square)
        pad_value: Value for padding (default 0.0)
    
    Returns:
        tensor: Preprocessed image tensor (1, target_size, target_size)
        original_tensor: Original image as tensor (1, H, W)
        scale: Scale factor applied to original coordinates
    """
    # Convert PIL to numpy array
    img_np = np.array(pil_image, dtype=np.float32)
    
    # Normalize to [0, 1]
    if img_np.max() > 1.0:
        img_np = img_np / 255.0
    
    # Convert to tensor and add channel dimension
    img_tensor = torch.from_numpy(img_np).unsqueeze(0)  # (1, H, W)
    
    # Letterbox: preserve aspect ratio and pad to square
    _, h, w = img_tensor.shape
    scale = min(target_size / h, target_size / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    
    # Resize
    if (new_h, new_w) != (h, w):
        img_resized = torch.nn.functional.interpolate(
            img_tensor.unsqueeze(0),
            size=(new_h, new_w),
            mode='bilinear',
            align_corners=False
        ).squeeze(0)
    else:
        img_resized = img_tensor
    
    # Pad to square
    pad_h = target_size - new_h
    pad_w = target_size - new_w
    top = pad_h // 2
    left = pad_w // 2
    
    padded = torch.full((1, target_size, target_size), pad_value, dtype=img_tensor.dtype)
    padded[:, top:top+new_h, left:left+new_w] = img_resized
    
    return padded, img_tensor, scale

# Apply preprocessing
imgsz = 512
preprocessed_tensor, original_tensor, scale_factor = preprocess_thermal_image(thermal_pil, target_size=imgsz)

print(f"Preprocessed tensor shape: {preprocessed_tensor.shape}")
print(f"Scale factor: {scale_factor:.4f}")
print(f"Value range: [{preprocessed_tensor.min():.4f}, {preprocessed_tensor.max():.4f}]")

# Add batch dimension and move to device
input_batch = preprocessed_tensor.unsqueeze(0).to(device)  # (1, 1, 512, 512)
print(f"Input batch shape: {input_batch.shape}")
print(f"Input batch device: {input_batch.device}")

## Section 4: Preprocess Thermal Image

Normalize and transform the thermal image to the required input format (single channel, correct dimensions).

In [5]:
# Specify thermal image path
thermal_images_dir = project_root / 'FLIR_ADAS_v2' / 'images_thermal_val' / 'data'

# Get list of available thermal images
available_images = list(thermal_images_dir.glob('*.jpg'))
print(f"Found {len(available_images)} thermal images")

# Select first image (or change index as needed)
selected_image_path = available_images[0]
print(f"Selected image: {selected_image_path.name}")

# Load image using PIL
thermal_pil = Image.open(selected_image_path)
thermal_pil = thermal_pil.convert('L')  # Ensure single channel (grayscale)
print(f"Original image size: {thermal_pil.size}")
print(f"Image mode: {thermal_pil.mode}")

Found 1144 thermal images
Selected image: video-kBvcuDMtYv2Z4kmXi-frame-000572-xo7MRdBzLhPXfbGsT.jpg
Original image size: (640, 512)
Image mode: L


## Section 3: Prepare Thermal Image Input

Load a thermal image from the FLIR dataset.

In [4]:
# Load the Distilled Checkpoint
checkpoint_path = project_root / 'checkpoints_distill' / 'distill_epoch_5.pth'

if checkpoint_path.exists():
    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # The checkpoint may contain model state dict directly or wrapped in 'model_state_dict'
    if isinstance(checkpoint, dict):
        if 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
            print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
            print(f"Loss: {checkpoint.get('loss', 'N/A')}")
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint
    
    # Load state dict into model
    student_model.load_state_dict(state_dict, strict=False)
    print("✓ Checkpoint loaded successfully")
else:
    print(f"⚠ Warning: Checkpoint not found at {checkpoint_path}")
    print("Using randomly initialized weights")

# Move model to device
student_model = student_model.to(device)
print(f"✓ Model moved to {device}")

Loading checkpoint from: /home/aryan_s2/Detection_with_Distillation/checkpoints_distill/distill_epoch_5.pth
✓ Checkpoint loaded successfully
✓ Model moved to cuda


## Section 2: Load Model Architecture and Weights

Define the student model architecture and load the trained weights from the distilled checkpoint.

In [3]:
# Import the Student Model
from Student.yolov8_thermal import yolov8s_thermal

# Create student model with 3 classes (person, car, bicycle)
num_classes = 3
student_model = yolov8s_thermal(num_classes=num_classes)
print(f"Student Model created with {num_classes} classes")
print(f"Model architecture: YOLOv8-Small Thermal")
print(f"Total parameters: {sum(p.numel() for p in student_model.parameters()):,}")

Student Model created with 3 classes
Model architecture: YOLOv8-Small Thermal
Total parameters: 22,137,772


## Section 1: Import Required Libraries

Import PyTorch, NumPy, Matplotlib, and PIL for image processing and model inference.

In [2]:
# Import Required Libraries
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from pathlib import Path
from PIL import Image
import json
import sys
import os

# Add project paths
project_root = Path('/home/aryan_s2/Detection_with_Distillation')
sys.path.insert(0, str(project_root))

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
CUDA available: True
GPU: NVIDIA RTX A6000


# Thermal Image Forward Pass Inference

This notebook performs forward passes on thermal images using the distilled YOLOv8 student model for object detection. It includes loading the trained checkpoint, preprocessing thermal images, running inference, and visualizing detection results.